In [2]:
from tifffile import imread
from extract_features_functions_2026 import *
from functions_2026 import *

In [3]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))


Num GPUs Available:  1


In [4]:
######### INPUTS ###########
# List of WSI paths
WSI_paths = [
    r'\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\dataset_224_40k_48_slides\train\images'
    ]

date = '4_3_2026' # date of the segmentation

saveMASKS =0 # save segmented cells masks
target_resolution=1 # mask resolution 1->10x
RGBfeatures=1 # do you want to extract rgb features from contours 1-yes,0-no
gj=1 # do the geojson countours of the segmented cells, 1 if yes, 0 if no
prob_maps=1 # save probability maps, 1 if yes, 0 if no
inverse=0

# Change this to the model you want to load
# model_name = "monkey"  #"pdac" "panin" "panin_healthy"  "monkey"  "fallopian_tube" ""
model_name = "cross_fetal_species"  #"pdac" "panin" "panin_healthy" "fallopian_tube" "cross_fetal_species" "cross_fetal_species_alcian_blue"

############################
for WSI_path in WSI_paths:
    # Define output paths
    out_pth = os.path.join(WSI_path, f'StarDist_{date}_{model_name}')
    out_pth_json = os.path.join(out_pth, 'json')
    out_pth_tif = os.path.join(out_pth, 'stardist_masks')
    output_pixres = os.path.join(WSI_path,'segmentation_analysis', 'pix_res_info')
    outpthpkl = os.path.join(out_pth, 'stardist_feature_df_pickles')
    outpthmat=os.path.join(outpthpkl,'mat')
    out_pth_contours = os.path.join(out_pth_json, 'geojsons', '32_polys_20x')
    out_pth_prob = os.path.join(out_pth, 'probability_maps')

    # Create directories if they don't exist
    for path in [out_pth, out_pth_json, out_pth_tif, output_pixres, outpthpkl, outpthmat, out_pth_contours, out_pth_prob]:
        os.makedirs(path, exist_ok=True)
    
    print(out_pth_json)

\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\dataset_224_40k_48_slides\train\images\StarDist_4_3_2026_cross_fetal_species\json


In [5]:
# model = load_model(model_path)
# model = load_selected_models_AF(model_name)
model = load_selected_models_folder_AF(model_name)
# model = load_selected_model(model_name)

base_model.py (198): output path for model already exists, files may be overwritten: C:\Users\Andre\cursor_projects\Convnext_stardist\stardist\models\cross_fetal_species\offshoot_model


Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.524214, nms_thresh=0.3.
Overriding defaults: Thresholds(prob=0.5242143538251474, nms=0.3) 

Loaded cross_fetal_species model successfully from repo.


In [ ]:
json_pth_list = get_sorted_files(out_pth_json, '.json')
# for WSI_path in WSI_paths[0:1]:
for WSI_path in WSI_paths:
    WSIs = get_sorted_files(WSI_path, '.ndpi', '.svs','.png')
    output_pixres=os.path.join(WSI_path,'segmentation_analysis', 'pix_res_info')
    out_pth_json = os.path.join(WSI_path, f'StarDist_{date}_{model_name}', 'json')
    out_pth_tif= os.path.join(WSI_path, f'StarDist_{date}_{model_name}', 'stardist_masks')
    out_pth_prob = os.path.join(WSI_path, f'StarDist_{date}_{model_name}', 'probability_maps')
    os.makedirs(out_pth_prob, exist_ok=True)
    extract_and_save_pixel_sizes(WSI_path, output_pixres)
    if prob_maps == 1:
        # Loads all tiles into RAM once, then predict_instances() per tile (same as
        # segment_and_save_JSONs_prob_maps, but no re-read from NAS per frame).
        # n_io_workers: parallel load + background JSON/prob TIFF writes.
        segment_and_save_JSONs_prob_maps_batched(
            WSIs, model, out_pth_json, out_pth_prob, inverse,
            step_seg=1, n_io_workers=8,
        )
    elif saveMASKS == 1:
        segment_and_save_JSONs_and_masks(WSIs, model, out_pth_json, out_pth_tif, output_pixres, target_resolution, inverse, step_seg=50)
    else:
        segment_and_save_JSONs(WSIs, model, out_pth_json, out_pth_tif, inverse, step_seg=1)


Skipping 250 already-processed tiles.
Tiles to process : 50388
Inference        : single-tile predict_instances (in-RAM images)
I/O workers      : 8

Loading all 50388 tiles into RAM using 8 threads...


Loading: 100%|██████████| 50388/50388 [03:04<00:00, 272.89tile/s]

Loaded in 185.7s — stacking...


Array shape: (50388, 224, 224, 3)  (30.3 GB)


Segment:   0%|          | 0/50388 [00:00<?, ?tile/s]base.py (716): Setting sparse to False because return_predict is True


In [ ]:
ds_amt = 1  #20x; 1/0.4416 #10x; 2/0.4416  #5x; 4 #2.5x

for WSI_path in WSI_paths:
    out_pth_json = os.path.join(WSI_path, f'StarDist_{date}_{model_name}', 'json')
    # Call the function to create GeoJSON contours
    make_geojson_contours(out_pth_json, 
                          gj=gj, 
                          ds_amt=ds_amt, 
                          classification_name='Nuclei', 
                          classification_color=[97, 214, 59],
                          stepsize=1)

In [ ]:
for WSI_path in WSI_paths:
    print(WSI_path)
    # Get the files
    WSIs = get_sorted_files(WSI_path, '.ndpi','.svs')
    out_pth_json = os.path.join(WSI_path, f'StarDist_{date}_{model_name}', 'json')
    jsons = get_sorted_files(out_pth_json, '.json')
    output_pixres=os.path.join(WSI_path,'segmentation_analysis', 'pix_res_info')
    pixel_res_files = get_sorted_files(output_pixres, '.mat')
    # crop_mats = get_sorted_files(pth_crop_data, '.mat')

    print(f"Number of WSIs: {len(WSIs)}")
    print(f"Number of JSON files: {len(jsons)}")
    print(f"Number of Pixel Res Files: {len(pixel_res_files)}")

In [ ]:
# CHECK IF FILES MATCH
if check_file_alignment(jsons, WSIs, pixel_res_files):
    print("All files match.")
else:
    print("There is a mismatch in the files.")

In [ ]:
WSIs = get_sorted_files(WSI_path, '.ndpi','.svs')
WSIs[0]

In [ ]:
# EXTRACTS FEATURES FROM JSON CONTOURS AND SAVES THEM IN A PICKLE DATAFRAME
for WSI_path in WSI_paths:
    WSIs = get_sorted_files(WSI_path, '.ndpi','.svs')
    out_pth_json = os.path.join(WSI_path, f'StarDist_{date}_{model_name}', 'json')
    jsons = get_sorted_files(out_pth_json, '.json')
    out_pth = os.path.join(WSI_path, f'StarDist_{date}_{model_name}')
    outpthpkl = os.path.join(out_pth, 'stardist_feature_df_pickles')
    if RGBfeatures == 1:
        # computes also the RGB average intensities inside of each contour
        make_RGBfeatures_df_pkl_from_contours_jsons(jsons, WSIs, outpthpkl)
    else:
        make_features_df_pkl_from_contours_jsons(jsons, WSIs, outpthpkl)
    

In [ ]:
# MAKE LIST OF FEATURES DF PKL FILES AND SAVES THEM AS MAT FILES 
for WSI_path in WSI_paths:
    out_pth = os.path.join(WSI_path, f'StarDist_{date}_{model_name}')
    outpthpkl = os.path.join(out_pth, 'stardist_feature_df_pickles')
    outpthmat = os.path.join(outpthpkl,'mat')
    dfs = get_sorted_files(outpthpkl, '.pkl')
    featuresdf_pkl2mat(outpthpkl, outpthmat, dfs)

In [ ]:
import os
WSI_paths = [
    # r'\\kittyserverdw\Andre_kit\data\BTC\H10',
    # r'\\kittyserverdw\Andre_kit\data\BTC\H11',
    # r'\\kittyserverdw\Andre_kit\data\BTC\H19',
    r'\\kittyserverdw\Andre_kit\data\monkey_fetus\Turtles\Tscripta239_Paul'
    ]
for WSI_path in WSI_paths:
    print(WSI_path)
    # Get the files
    WSIs = get_sorted_files(WSI_path, '.ndpi','.svs')
    out_pth_json = os.path.join(WSI_path, f'StarDist_{date}_{model_name}', 'json')
    jsons = get_sorted_files(out_pth_json, '.json')
    output_pixres=os.path.join(WSI_path,'segmentation_analysis', 'pix_res_info')
    pixel_res_files = get_sorted_files(output_pixres, '.mat')
    out_pth = os.path.join(WSI_path, f'StarDist_{date}_{model_name}')
    outpthpkl = os.path.join(out_pth, 'stardist_feature_df_pickles')
    crop_info_matfile = os.path.join(WSI_path, 'chop', 'metadata')
    dfs = get_sorted_files(outpthpkl, '.pkl')
    outpthmat = os.path.join(outpthpkl,'mat')
    outpthmatchop=os.path.join(outpthmat,'chop')
    if not os.path.exists(outpthmatchop):
        os.makedirs(outpthmatchop)

    pth_imhe_chop = os.path.join(WSI_path, 'chop', '1x')
    # pth_imhe_chop = os.path.join(WSI_path, 'chop', '1x_cleaned')

    featuresdf_pkl2mat_chop(outpthpkl,
                            outpthmatchop,
                            dfs,
                            crop_info_matfile,
                            output_pixres,
                            9.9988, #8 ,#7.9176
                            9.9988, #8 #,7.9213
                            pth_imhe_chop
                            )

In [ ]:
# crop_info_matfile = r'\\169.254.138.20\Andre_expansion\data\monkey_fetus\Lucie_mouse_fetuses\MM11\chop\metadata'
crop_info_matfile = r'\\kittyserverdw\Andre_kit\data\monkey_fetus\Turtles\Tscripta239_Paul\chop\metadata'
# crop_info_matfile = r'\\10.162.80.16\Andre_expansion\data\monkey_fetus\Alligator\chop\metadata'
dfs = get_sorted_files(outpthpkl, '.pkl')

# featuresdf_pkl2mat_chop(outpthpkl, outpthmatchop, dfs, crop_info_matfile,output_pixres, 8, 8) #7.9176,7.9213
featuresdf_pkl2mat_chop(outpthpkl, outpthmatchop, dfs, crop_info_matfile,output_pixres, 9.9988, 9.9988) #7.9176,7.9213